# 1. Príprava dát

### 1.1 Načítanie a zlúčenie dát z jednotlivých vĺn

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")

FILES = {
    "Wuhan": DATA_DIR / "wuhan.csv",
    "Alfa": DATA_DIR / "alfa.csv",
    "Delta": DATA_DIR / "delta.csv",
    "Omikron": DATA_DIR / "omikron.csv",
}

def read_csv_auto(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", encoding="utf-8", low_memory=False)

dfs = []
for wave, file in FILES.items():
    df = read_csv_auto(file)
    df["Vlna"] = wave
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data.to_csv(DATA_DIR / "cdi_merged_dataset.csv", index=False, sep=";", encoding="utf-8")

ERROR! Session/line number was not unique in database. History logging moved to new session 535


### 1.2 Odstránenie neštruktúrovaných textových polí

In [2]:
DROP_TEXT_COLS = [
    "Poradie", "Meno", "Kód príjmu", "HLN Dg.", "Diagnózy", "DRG výkony", "Liečba",
    "SVLZ správy", "Mikrobiológia ", "Epikríza", "Terajšie ochorenie", "Dôvod hospitalizácie",
    "Objektívny nález", "Osobná anamnéza", "Lieková anamnéza", "Návyková anamnéza",
    "Epidemiologická anamnéza", "Unnamed: 23"
]

data = data.drop(columns=[c for c in DROP_TEXT_COLS if c in data.columns], errors="ignore")

### 1.3 Odhalenie chýbajúcich hodnôt 

In [3]:
missing_summary = (
    data.isna().sum().reset_index()
    .rename(columns={"index": "Atribút", 0: "Počet_prázdnych"})
)
missing_summary["Percento_prázdnych"] = (missing_summary["Počet_prázdnych"] / len(data) * 100).round(2)
missing_summary = missing_summary.sort_values(by="Počet_prázdnych", ascending=False)

print(missing_summary.to_string(index=False))

                                                   Atribút  Počet_prázdnych  Percento_prázdnych
                                                S-Chol max             3710               96.41
                                                S-Chol min             3710               96.41
                                               S-Chol last             3710               96.41
                                              S-Chol first             3710               96.41
                                                 S-IgA min             3657               95.04
                                                 S-IgA max             3657               95.04
                                                S-IgA last             3657               95.04
                                               S-IgA first             3657               95.04
                                              S-Ig M first             3648               94.80
                                        

### 1.4 Konverzia numerických hodnôt uložených ako text

In [4]:
EXCLUDE_FROM_NUMERIC_PARSE = {
    "A04.7", "Vlna", "Dátum príjmu", "Dátum prepustenia", "Pohlavie"
}

object_cols = [
    col for col in data.select_dtypes(include=["object"]).columns
    if col not in EXCLUDE_FROM_NUMERIC_PARSE
]

def to_float(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(r"[^0-9\.\-]", "", regex=True)
        .replace("", np.nan),
        errors="coerce"
    )

for col in object_cols:
    data[col] = to_float(data[col])

print(f"Konvertovaných object stĺpcov na float: {len(object_cols)}")

Konvertovaných object stĺpcov na float: 181


C:\Users\pavel\AppData\Local\Temp\ipykernel_12480\1655389322.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace("", np.nan),


### 1.5 Ošetrenie extrémov pre SatO2

In [5]:
if "SatO2 %" in data.columns:
    data.loc[data["SatO2 %"] < 30, "SatO2 %"] = np.nan
    data.loc[data["SatO2 %"] > 1000, "SatO2 %"] = np.nan

    mask = (data["SatO2 %"] > 300) & (data["SatO2 %"] <= 1000)
    data.loc[mask, "SatO2 %"] = data.loc[mask, "SatO2 %"] / 10

### 1.6 Mapovanie pohlavia na binárnu hodnotu

In [6]:
if "Pohlavie" in data.columns:
    data["Pohlavie"] = data["Pohlavie"].map({"Muž": 1, "Žena": 0})

### 1.7 Konverzia True/False stĺpcov na 0/1

In [7]:
def to_01(s: pd.Series) -> pd.Series:
    s_str = s.astype(str).str.strip().str.lower()
    out = s_str.map({"true": 1, "false": 0})
    out[s_str.isin(["nan", "", "none"])] = np.nan
    return out

for col in data.columns:
    uniq = (
        data[col].dropna().astype(str).str.strip().str.lower().unique()
    )
    if len(uniq) > 0 and set(uniq).issubset({"true", "false"}):
        data[col] = to_01(data[col])

### 1.8 Výpočet dĺžky hospitalizácie a odstránenie dátumov

In [8]:
data = data.copy()
if "Dátum príjmu" in data.columns and "Dátum prepustenia" in data.columns:
    data["Dátum príjmu"] = pd.to_datetime(data["Dátum príjmu"], format="%d.%m.%Y", errors="coerce")
    data["Dátum prepustenia"] = pd.to_datetime(data["Dátum prepustenia"], format="%d.%m.%Y", errors="coerce")

    data["Dĺžka hospitalizácie"] = (data["Dátum prepustenia"] - data["Dátum príjmu"]).dt.days
    data = data.drop(columns=["Dátum príjmu", "Dátum prepustenia"], errors="ignore")

### 1.9 Zakódovanie vlny (kategória → číslo)

In [9]:
if "Vlna" in data.columns:
    data["Vlna"] = data["Vlna"].map({"Wuhan": 1, "Alfa": 2, "Delta": 3, "Omikron": 4}).fillna(0).astype(int)

### 1.10 Uloženie datasetu

In [10]:
data.to_csv(DATA_DIR / "cdi_base_clean.csv", sep=";", index=False, encoding="utf-8")
print("Dataset uložený ako 'cdi_base_clean.csv'")

Dataset uložený ako 'cdi_base_clean.csv'
